# Bigram Full Pipeline Results Viewer
ดูผลลัพธ์จาก `bigram_full_pipeline.py`

ประกอบด้วย:
- metrics summary ของแต่ละคู่แพลตฟอร์ม (SGD/RF)
- กราฟเปรียบเทียบ Top-1 Accuracy และ Candidate Recall
- ตัวอย่าง prediction ที่ถูกและผิด


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


In [ ]:
base = 'data/processed/bigram_pipeline_results'
metrics_path = f'{base}/bigram_pipeline_metrics_summary.csv'
metrics = pd.read_csv(metrics_path)
metrics


In [ ]:
# Pivot table เปรียบเทียบ SGD vs RF
pivot_acc = metrics.pivot(index='pair_name', columns='model', values='top1_accuracy').sort_index()
pivot_rec = metrics.pivot(index='pair_name', columns='model', values='candidate_recall_topk').sort_index()
print('Top-1 Accuracy')
display(pivot_acc)
print('Candidate Recall@top-k')
display(pivot_rec)

if set(['SGD','RF']).issubset(set(pivot_acc.columns)):
    diff = (pivot_acc['SGD'] - pivot_acc['RF']).rename('SGD_minus_RF')
    print('Accuracy difference (SGD - RF)')
    display(diff.to_frame())


In [ ]:
# Plot: accuracy และ candidate recall
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for model, g in metrics.groupby('model'):
    g = g.sort_values('pair_name')
    axes[0].plot(g['pair_name'], g['top1_accuracy'], marker='o', label=model)
    axes[1].plot(g['pair_name'], g['candidate_recall_topk'], marker='o', label=model)

axes[0].set_title('Top-1 Accuracy by Pair')
axes[0].set_ylabel('accuracy')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

axes[1].set_title('Candidate Recall@top-k by Pair')
axes[1].set_ylabel('recall')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Load prediction files
pred_files = {
    'pairs_instagram_googleplus': {
        'SGD': f'{base}/pairs_instagram_googleplus_top1_predictions_sgd.csv',
        'RF': f'{base}/pairs_instagram_googleplus_top1_predictions_rf.csv',
    },
    'pairs_twitter_googleplus': {
        'SGD': f'{base}/pairs_twitter_googleplus_top1_predictions_sgd.csv',
        'RF': f'{base}/pairs_twitter_googleplus_top1_predictions_rf.csv',
    },
    'pairs_twitter_instagram': {
        'SGD': f'{base}/pairs_twitter_instagram_top1_predictions_sgd.csv',
        'RF': f'{base}/pairs_twitter_instagram_top1_predictions_rf.csv',
    },
}

preds = {}
for pair_name, mm in pred_files.items():
    preds[pair_name] = {}
    for model, path in mm.items():
        preds[pair_name][model] = pd.read_csv(path)

for pair_name, mm in preds.items():
    for model, df in mm.items():
        print(f'{pair_name} | {model}: rows={len(df):,}')


In [ ]:
# ดูตัวอย่างเคสถูกและเคสผิด
pair_name = 'pairs_twitter_instagram'   # เปลี่ยนได้
model = 'SGD'                            # 'SGD' หรือ 'RF'
df = preds[pair_name][model]

show_cols = [
    'source_profile_id', 'predicted_profile_id', 'is_correct_top1',
    'predicted_match_probability', 'uu_sim', 'un_sim', 'ub_sim', 'true_candidate_in_topk'
]

print('Correct examples')
display(df[df['is_correct_top1'] == 1][show_cols].head(10))

print('Wrong examples')
display(df[df['is_correct_top1'] == 0][show_cols].head(10))


In [ ]:
# วิเคราะห์เคสที่หาไม่เจอใน candidate top-k
rows = []
for pair_name, mm in preds.items():
    for model, df in mm.items():
        rows.append({
            'pair_name': pair_name,
            'model': model,
            'missing_true_in_topk': int((df['true_candidate_in_topk'] == 0).sum()),
            'total': len(df),
        })
pd.DataFrame(rows).sort_values(['pair_name','model'])
